In [1]:
import os
import numpy as np


path = "lsa64"
files = os.listdir(path)
print(len(files))
labels = [int(f[:3]) for f in files]
num_classes = len(set(labels))
poses = [np.load("lsa64/" + f) for f in files]

3200


In [2]:
# check min and max frame amount of keypoints
min_frames = min(p.shape[0] for p in poses)
max_frames = max(p.shape[0] for p in poses)
print(min_frames, max_frames)

14 201


In [3]:
# add padding to the end of each pose to make them all the same length
padded_poses = [
    np.pad(p, ((0, max_frames - p.shape[0]), (0, 0), (0, 0))) for p in poses
]

In [12]:
# do a stratified split of the data
from sklearn.model_selection import train_test_split

# do for both the labels and the poses
y_train, y_test, x_train, x_test = train_test_split(
    labels, poses, stratify=labels, test_size=0.2, random_state=420, shuffle=True
)
print(len(y_train), len(y_test))

2560 640


In [18]:
import pandas as pd

df = pd.DataFrame(y_test, columns=["Label"])
df["ID"] = df.index
# usage should be half Public and half Private, stratified by label
df.sort_values("Label", inplace=True)
df.reset_index(drop=True, inplace=True)
df["Usage"] = "Public"
df.loc[df.index % 2 == 0, "Usage"] = "Private"
df = df[["ID", "Usage", "Label"]]
df

,ID,Usage,Label
0,461,Private,1
1,194,Public,1
2,264,Private,1
3,191,Public,1
4,558,Private,1
...,...,...,...
635,122,Public,64
636,328,Private,64
637,550,Public,64
638,579,Private,64


In [27]:
df.to_csv("data/test/labels.csv", index=False)

In [28]:
for id in df["ID"]:
    np.save(f"data/test/poses/{id}.npy", x_test[id])

In [29]:
df_train = pd.DataFrame(y_train, columns=["Label"])
df_train["ID"] = df_train.index
df_train = df_train[["ID", "Label"]]
df_train.to_csv("data/train/labels.csv", index=False)

In [30]:
for id in df_train["ID"]:
    np.save(f"data/train/poses/{id}.npy", x_train[id])